# 7 — Graphes : explorer sans se perdre


## Fil conducteur

Le backtracking parcourait un arbre : chaque branche descendait vers de nouveaux
choix. Un graphe est plus général : plusieurs chemins peuvent rejoindre le même
sommet et un chemin peut revenir en arrière. Nous ajouterons donc une seule idée,
l'ensemble des sommets déjà vus, puis comparerons deux ordres d'exploration.


## 1. Sommets, arêtes et chemins

Un **sommet** représente un objet : ville, personne, page ou état. Une **arête**
relie deux sommets. Dans un graphe **orienté**, l'arête A → B possède un sens :
elle n'autorise pas automatiquement B → A.

Un **chemin** est une suite de sommets reliés dans le bon sens. Sa longueur est
ici le nombre d'arêtes utilisées. Un sommet est **accessible** depuis A s'il
existe au moins un chemin de A jusqu'à lui.

Un **cycle** est un chemin qui revient vers un sommet déjà rencontré. Un graphe
est dit **non pondéré** lorsque les arêtes n'ont pas de coûts différents : dans
ce cours, chaque arête compte simplement pour une étape.

### Exemple illustré — sommets, arêtes, chemin et cycle

```text
A ─→ B ─→ C
↑         │
└─────────┘
```

`A`, `B`, `C` sont des sommets. Les flèches sont des arêtes. `A → B → C` est un
chemin ; le retour `C → A` forme un cycle.


## 2. Représenter les voisins

Une liste d'adjacence associe chaque sommet à ses voisins sortants :

| sommet | voisins |
|---|---|
| A | B, C |
| B | D |
| C | D |
| D | A |
| Z | aucun |

En Python, on peut écrire
<code>{"A": ["B", "C"], "B": ["D"], "C": ["D"], "D": ["A"], "Z": []}</code>.
Le lien D → A crée un cycle. Z est isolé.

### Exemple illustré — la liste d’adjacence répond « quels voisins ? »

| Sommet | Voisins sortants |
|---|---|
| A | B, C |
| B | D |
| C | D |
| D | aucun |

Le dictionnaire associe chaque sommet à la liste des sommets atteignables en une
arête.


## 3. Pourquoi mémoriser les sommets vus ?

Sans mémoire, suivre A → B → D → A recommencerait indéfiniment. Un ensemble
<code>vus</code> répond à la question « ce sommet a-t-il déjà été découvert ? ».

On marque un sommet quand on décide de l'explorer. Chaque sommet n'est alors
traité qu'une fois. Ce principe est commun à DFS et BFS ; ce qui change est
l'ordre des sommets encore à traiter.

### Exemple illustré — `vus` bloque un cycle

```text
A → B → A
vus = {A}       après A
vus = {A, B}    après B
retour vers A   A est déjà vu → ne pas recommencer
```

Sans cet ensemble, le parcours alternerait indéfiniment entre A et B.




*Image — Marquer un sommet casse les cycles du parcours.*


## 4. DFS : aller au fond avant de revenir

DFS signifie **Depth-First Search**, ou parcours en profondeur. Depuis A, on
choisit un voisin, puis un voisin de ce voisin, jusqu'à ne plus pouvoir avancer.
On revient alors au dernier choix non terminé.

La récursion utilise naturellement la pile d'appels. Une pile explicite est
aussi possible. DFS répond bien à la question d'accessibilité, mais le premier
chemin rencontré n'est pas forcément le plus court.

### Exemple illustré — DFS termine une branche avant la suivante

```text
A
├─ B
│  └─ D   ← exploré avant C
└─ C
```

L’ordre est `A, B, D, C`. Le parcours revient vers A seulement lorsque B ne possède
plus de voisin inexploré.


In [ ]:
graphe = {"A": ["B", "C"], "B": ["D"], "C": ["D"], "D": ["A"], "Z": []}

def accessibles_dfs(graphe, depart):
    vus = set()
    ordre = []

    def visiter(sommet):
        vus.add(sommet)
        ordre.append(sommet)
        for voisin in graphe.get(sommet, []):
            if voisin not in vus:
                visiter(voisin)

    visiter(depart)
    return ordre

assert accessibles_dfs(graphe, "A") == ["A", "B", "D", "C"]


La trace est A, B, D, puis retour vers A pour explorer C. Le mot **retour**
désigne ici la reprise d'un appel suspendu, exactement comme dans les chapitres
sur la récursion et le backtracking.


## 5. BFS : explorer une couche après l'autre

BFS signifie **Breadth-First Search**, ou parcours en largeur. Il utilise une
file : les premiers sommets découverts sont les premiers traités.

- Couche 0 : le départ A.
- Couche 1 : B et C, atteints en une arête.
- Couche 2 : D, atteint en deux arêtes.

On marque un voisin dès son insertion dans la file. Sinon B et C pourraient
insérer D chacun de leur côté avant que D soit traité.

### Exemple illustré — BFS avance par couches

```text
couche 0 :        A
couche 1 :      B   C
couche 2 :        D
```

La file traite tous les sommets à une arête de A avant ceux situés à deux arêtes.




*DFS suit une branche ; BFS suit les couches de distance.*


In [ ]:
from collections import deque

def distances_bfs(graphe, depart):
    distances = {depart: 0}
    file = deque([depart])
    while file:
        sommet = file.popleft()
        for voisin in graphe.get(sommet, []):
            if voisin not in distances:
                distances[voisin] = distances[sommet] + 1
                file.append(voisin)
    return distances

assert distances_bfs(graphe, "A") == {"A": 0, "B": 1, "C": 1, "D": 2}


### Trace visuelle de la file BFS

Pour le graphe du cours, en partant de `A` :

| File avant retrait | Sommet traité | Nouveaux sommets ajoutés | Distances connues |
|---|:---:|---|---|
| `[A]` | A | B, C | A:0, B:1, C:1 |
| `[B, C]` | B | D | A:0, B:1, C:1, D:2 |
| `[C, D]` | C | aucun, D est déjà vu | inchangées |
| `[D]` | D | aucun, A est déjà vu | inchangées |

Marquer un sommet dès son ajout à la file évite de l’ajouter plusieurs fois avant
même qu’il soit traité.

## 6. Pourquoi la première distance BFS est minimale

BFS traite les couches dans l'ordre 0, 1, 2, etc. Lorsqu'un sommet de distance d
ajoute un voisin inconnu, il construit un chemin de longueur d + 1.

S'il existait un chemin plus court vers ce voisin, son avant-dernier sommet
appartiendrait à une couche plus petite, déjà traitée. Le voisin aurait donc été
découvert plus tôt. Sa première découverte donne bien un nombre minimal d'arêtes.

Cette preuve suppose des arêtes non pondérées, ou toutes de même coût. Avec des
coûts différents, « moins d'arêtes » ne signifie plus « coût total plus petit ».

Pour reconstruire le chemin, on mémorise aussi le **parent** d'un sommet : c'est
le sommet depuis lequel il a été découvert pour la première fois. Remonter les
parents depuis l'arrivée ramène jusqu'au départ.

### Exemple illustré — reconstruire avec les parents

```text
parent[B] = A
parent[D] = B

D ← B ← A    puis inversion    A → B → D
```

Chaque parent indique depuis quel sommet la première découverte a eu lieu.




*L’ensemble des sommets vus empêche les retours en cycle ; les parents permettent de reconstruire le chemin.*

## 7. Comprendre O(V + E)

On note V le nombre de sommets et E le nombre d'arêtes. Chaque sommet accessible
est marqué une fois : au plus V traitements. Chaque liste de voisins est parcourue
une fois : au total, E entrées pour un graphe orienté.

Le travail est donc O(V + E). Le signe + vient de deux quantités différentes,
pas de deux boucles arbitraires. Les ensembles, distances, parents et la file
peuvent conserver jusqu'à V sommets : O(V) mémoire.

### Exemple illustré — compter sommets et arêtes une fois

Dans `A → B`, `A → C`, `B → D`, `C → D` :

| Quantité | Valeur | Travail du parcours |
|---|---:|---:|
| sommets `V` | 4 | 4 traitements |
| arêtes `E` | 4 | 4 inspections |

Le total suit `V + E`, car chaque sommet et chaque arête sont considérés un nombre
constant de fois.


## Exercices progressifs

Commence par lire directement la représentation, puis écris un BFS, enfin ajoute
les parents nécessaires à la reconstruction d'un chemin.


### Palier A — Nombre d'arêtes sortantes

Le nombre de voisins sortants est appelé degré sortant. Renvoie ce nombre pour un sommet. Un sommet absent possède
zéro voisin.


In [ ]:
def degre_sortant(graphe, sommet):
    return len(graphe.get(sommet, []))


In [ ]:
assert degre_sortant({"A": ["B", "C"]}, "A") == 2
assert degre_sortant({}, "Z") == 0


**Pourquoi cela fonctionne.** La liste d'adjacence contient directement les
arêtes sortantes. <code>get</code> fournit une liste vide si le sommet est absent.


### Palier B — Ensemble accessible par BFS

Renvoie l'ensemble des sommets accessibles depuis le départ. Marque lors de
l'insertion dans la file.


In [ ]:
def accessibles_bfs(graphe, depart):

    vus = {depart}
    file = deque([depart])
    while file:
        sommet = file.popleft()
        for voisin in graphe.get(sommet, []):
            if voisin not in vus:
                vus.add(voisin)
                file.append(voisin)
    return vus


In [ ]:
assert accessibles_bfs(graphe, "A") == {"A", "B", "C", "D"}
assert accessibles_bfs(graphe, "Z") == {"Z"}


**Pourquoi cela fonctionne.** L'ensemble contient exactement les sommets déjà
découverts. La file contient ceux dont les voisins restent à examiner.
Chaque sommet est inséré au plus une fois.


### Palier C — Un plus court chemin non pondéré

Renvoie une liste de sommets formant un plus court chemin, ou <code>None</code>
si l'arrivée est inaccessible. Enregistre le parent d'un sommet lors de sa
première découverte, puis reconstruis le chemin à l'envers.


In [ ]:
def plus_court_chemin(graphe, depart, arrivee):

    parents = {depart: None}
    file = deque([depart])
    while file:
        sommet = file.popleft()
        if sommet == arrivee:
            chemin = []
            while sommet != depart:
                chemin.append(sommet)
                sommet = parents[sommet]
            chemin.append(depart)
            return chemin[::-1]
        for voisin in graphe.get(sommet, []):
            if voisin not in parents:
                parents[voisin] = sommet
                file.append(voisin)
    return None


In [ ]:
assert plus_court_chemin(graphe, "A", "D") == ["A", "B", "D"]
assert plus_court_chemin(graphe, "A", "Z") is None
assert plus_court_chemin(graphe, "A", "A") == ["A"]
assert plus_court_chemin({None: ["A"], "A": []}, None, "A") == [None, "A"]


**Pourquoi cela fonctionne.** La première découverte BFS donne une distance
minimale. Le parent mémorise l'arête utilisée pour cette découverte. Remonter
les parents jusqu'au départ reconstruit donc un chemin minimal.


## Questions de compréhension

1. Pourquoi un graphe demande-t-il un ensemble de sommets vus ?
2. Quelle structure donne l'ordre DFS ? Quelle structure donne l'ordre BFS ?
3. Pourquoi marque-t-on un voisin avant de l'ajouter à la file ?
4. Pourquoi BFS ne garantit-il pas le chemin de coût minimal avec des poids différents ?


**Éléments de réponse.** Les cycles et chemins convergents feraient retraiter des
sommets. DFS utilise une pile, explicite ou récursive ; BFS utilise une file.
Marquer avant l'insertion empêche les insertions multiples. Avec des poids,
le nombre d'arêtes ne représente plus le coût total.


## À retenir et vérifier

La représentation définit les voisins. Le marquage évite les cycles. DFS explore
une branche avant les autres ; BFS explore les couches et donne les distances
minimales en nombre d'arêtes. Avec des listes d'adjacence, le coût est O(V + E).

**Étape suivante.** Le chapitre 8 présente les expressions régulières comme un
outil spécialisé pour les contrats textuels. Il peut être traité comme module
optionnel avant l'atelier final.
